In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
import torch

from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import precision_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from mealpy.swarm_based import PSO, BA, CSO, FA, ABC
from mealpy.evolutionary_based import GA
from mealpy.swarm_based.ACOR import OriginalACOR
from mealpy.utils.problem import FloatVar

warnings.filterwarnings("ignore")
print("cwd:", os.getcwd())

# ---------------------------
# Step 1 — Load dataset and BERT embeddings (cache to speed up)
# ---------------------------
def get_bert_train_test(cache_x="bert_emb.npy", cache_y="bert_labels.npy"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    if os.path.exists(cache_x) and os.path.exists(cache_y):
        X = np.load(cache_x)
        y = np.load(cache_y)
        print("Loaded cached embeddings.")
    else:
        print("Generating BERT embeddings...")
        paths = [
            "../sentiment labelled sentences/amazon_cells_labelled.txt",
            "../sentiment labelled sentences/imdb_labelled.txt",
            "../sentiment labelled sentences/yelp_labelled.txt",
        ]
        df = pd.concat([pd.read_csv(p, sep="\t", header=None, names=["text","label"]) for p in paths], ignore_index=True)

        tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        model = BertModel.from_pretrained("bert-base-uncased").to(device)
        model.eval()

        def encode(texts, bs=32):  # increase batch size for speed
            embeds = []
            with torch.no_grad():
                for i in range(0, len(texts), bs):
                    batch = texts[i:i+bs]
                    enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
                    out = model(**enc)
                    embeds.append(out.last_hidden_state.mean(dim=1).cpu())
            return torch.cat(embeds).numpy()

        X = encode(df["text"].tolist())
        y = df["label"].values
        np.save(cache_x, X)
        np.save(cache_y, y)
        print("Saved embeddings to disk.")

    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_test, y_train, y_test = get_bert_train_test()

# ---------------------------
# Step 2 — Objective function
# ---------------------------
def objective(solution, model_name):
    n_estimators      = int(solution[0])
    learning_rate     = float(solution[1])
    max_depth         = int(solution[2])
    n_neighbors       = int(solution[3])
    gamma_val         = float(solution[4])
    min_samples_leaf  = int(solution[5])
    max_features      = float(solution[6])
    ccp_alpha_val     = float(solution[7])

    try:
        if model_name == "DecisionTree":
            clf = DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                ccp_alpha=ccp_alpha_val,
                random_state=42
            )
        elif model_name == "GradientBoosting":
            clf = GradientBoostingClassifier(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=42
            )
        elif model_name in ["RandomForest", "RandomForest_Specificity"]:
            clf = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=42
            )
        elif model_name == "KNN":
            clf = KNeighborsClassifier(n_neighbors=n_neighbors)
        elif model_name == "SVM":
            clf = SVC(C=gamma_val, kernel='rbf', gamma='scale', random_state=42)
        elif model_name == "XGBoost":
            clf = XGBClassifier(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                random_state=42,
                use_label_encoder=False,
                eval_metric="logloss"
            )
        else:
            return 1.0  # unknown model

        # Use 2-fold CV for faster evaluation
        y_pred = cross_val_predict(clf, X_train, y_train, cv=2)

        if model_name == "RandomForest_Specificity":
            tn = int(((y_train == 0) & (y_pred == 0)).sum())
            fp = int(((y_train == 0) & (y_pred == 1)).sum())
            specificity = tn / (tn + fp + 1e-9)
            return -specificity
        else:
            prec = precision_score(y_train, y_pred, zero_division=0)
            return -prec

    except Exception as e:
        print(f"Error in {model_name}:", e)
        return 1.0

# ---------------------------
# Step 3 — Problem bounds
# ---------------------------
problem = {
    "bounds": [
        FloatVar(50, 300, 'n_estimators'),
        FloatVar(0.01, 0.3, 'learning_rate'),
        FloatVar(2, 12, 'max_depth'),
        FloatVar(1, 20, 'n_neighbors'),
        FloatVar(0.01, 10.0, 'gamma_or_C'),
        FloatVar(1, 10, 'min_samples_leaf'),
        FloatVar(0.5, 1.0, 'max_features'),
        FloatVar(0.0, 0.1, 'ccp_alpha')
    ],
    "minmax": "min",
    "obj_func": None
}

# ---------------------------
# Step 4 — Metaheuristic algorithms
# ---------------------------
algorithms = {
    "PSO": PSO.OriginalPSO(epoch=5, pop_size=10),
    "GA": GA.BaseGA(epoch=5, pop_size=10),
    "BA": BA.OriginalBA(epoch=5, pop_size=10),
    "ACO": OriginalACOR(epoch=5, pop_size=10),
    "CSO": CSO.OriginalCSO(epoch=5, pop_size=10),
    "FA": FA.OriginalFA(epoch=5, pop_size=10),
    "ABC": ABC.OriginalABC(epoch=5, pop_size=10)
}

models = ["GradientBoosting", "DecisionTree", "RandomForest", "KNN", "SVM", "XGBoost", "RandomForest_Specificity"]

# ---------------------------
# Step 5 — Run optimization (fast, safe defaults if failure)
# ---------------------------
results = {}
best_params = {}

for model_name in models:
    print(f"\n=== Optimizing {model_name} ===")
    for algo_name, optimizer in algorithms.items():
        print(f"  -> {algo_name} ...", end=" ")
        problem["obj_func"] = lambda sol, m=model_name: objective(sol, m)
        try:
            best_agent = optimizer.solve(problem)
            sol = best_agent.solution
            score = -best_agent.target.fitness
        except Exception as e:
            print("failed:", e)
            sol = [50,0.01,2,1,1.0,1,0.5,0.0]
            score = 0.0

        params = {
            "n_estimators": int(sol[0]),
            "learning_rate": round(sol[1],4),
            "max_depth": int(sol[2]),
            "n_neighbors": int(sol[3]),
            "gamma_or_C": round(sol[4],4),
            "min_samples_leaf": int(sol[5]),
            "max_features": round(sol[6],4),
            "ccp_alpha": round(sol[7],5)
        }

        results[f"{model_name}_{algo_name}"] = score
        best_params[f"{model_name}_{algo_name}"] = params
        print(f"done. score={score:.4f}")

# ---------------------------
# Step 6 — Display results
# ---------------------------
print("\nFinal results:")
for k, v in results.items():
    print(f"{k}: {v:.4f}, params: {best_params[k]}")

2025/10/11 03:39:25 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: OriginalPSO(epoch=5, pop_size=10, c1=2.05, c2=2.05, w=0.4)


cwd: C:\Users\HP\OneDrive\Desktop\Technical-seminar\Technical-seminar\first_draft
Using device: cpu
Loaded cached embeddings.

=== Optimizing GradientBoosting ===
  -> PSO ... 